<a href="https://colab.research.google.com/github/castrokelly/PPGIa/blob/main/trabalho_implementacao_1_determinantes_KellyCastro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pontifícia Universidade Católica do Paraná

**Aluna:** 40131556 - Kelly Christine Alvarenga de Castro (christine.kelly@pucpr.edu.br)  
**Mestrado / Turma:** PPGIa (404) 2026/02  
**Disciplina:** Fundamentos de Matemática Computacional (Turma ME)  
**Professor:** Dr. Vinícius Mourão Alves de Souza

**Link do Notebook no Github:** https://github.com/castrokelly/PPGIa/blob/main/trabalho_implementacao_1_determinantes_KellyCastro.ipynb

**Link do Notebook no Youtube:** https://youtu.be/6YEpccfGegg

# TRABALHO DE IMPLEMENTAÇÃO 1 - Cálculo do determinante

**Objetivo:** Implementar uma função que recebe uma matriz quadrada e retorna seu determinante, seguindo os métodos exigidos no enunciado: produto das diagonais na ordem 2, Regra de Sarrus na ordem 3 e Teorema de Laplace nas ordens 4 ou superiores. Os quatro testes principais usam as matrizes da aula **Week 5** e comparam a implementação com `np.linalg.det`.

## 1. Importação da biblioteca

Usaremos o NumPy para representar as matrizes, remover linhas e colunas e conferir os resultados. O cálculo da função própria será escrito explicitamente, sem chamar `np.linalg.det` dentro dela.


In [1]:
import numpy as np


## 2. Criação da função

A função verifica se a matriz é quadrada e escolhe o método pela sua ordem $n$:

- **Ordem 1:** $\det(A)=a_{11}$. Esse caso completa o tratamento de qualquer ordem positiva.
- **Ordem 2:** produto da diagonal principal menos o produto da secundária:
  $$\det(A)=a_{11}a_{22}-a_{12}a_{21}.$$
- **Ordem 3:** a Regra de Sarrus soma três produtos e subtrai os outros três:
  $$\det(A)=(a_{11}a_{22}a_{33}+a_{12}a_{23}a_{31}+a_{13}a_{21}a_{32})-(a_{13}a_{22}a_{31}+a_{11}a_{23}a_{32}+a_{12}a_{21}a_{33}).$$
- **Ordem 4 ou maior:** expansão de Laplace pela primeira linha:
  $$\det(A)=\sum_{j=1}^{n} a_{1j}(-1)^{1+j}\det(M_{1j}),$$
  em que $M_{1j}$ é a submatriz obtida ao retirar a primeira linha e a coluna $j$. O menor complementar é o número $\det(M_{1j})$, e o cofator é $C_{1j}=(-1)^{1+j}\det(M_{1j})$.

No Python, os índices começam em zero. Por isso, a primeira linha é `A[0, :]` e o sinal dos cofatores é `(-1) ** j`, produzindo $+, -, +, -, \ldots$. A chamada recursiva calcula o determinante da submatriz; a ordem diminui até alcançar um caso direto. Na expansão de uma matriz 4 × 4, os determinantes das submatrizes 3 × 3 são calculados por Sarrus.

**Base:** Week 5, pp. 17–27; Robbiano, seção 3.7, pp. 73–75 (pp. 92–94 do PDF).


In [2]:
def determinante(matriz):
    A = np.array(matriz, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A matriz deve ser quadrada.")
    if A.shape[0] == 0:
        raise ValueError("A matriz não pode ser vazia.")

    n = A.shape[0]

    # Caso de ordem 1.
    if n == 1:
        return A[0, 0]

    # Ordem 2: produto das diagonais.
    if n == 2:
        return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]

    # Ordem 3: Regra de Sarrus.
    if n == 3:
        soma_positiva = (
            A[0, 0] * A[1, 1] * A[2, 2]
            + A[0, 1] * A[1, 2] * A[2, 0]
            + A[0, 2] * A[1, 0] * A[2, 1]
        )
        soma_negativa = (
            A[0, 2] * A[1, 1] * A[2, 0]
            + A[0, 0] * A[1, 2] * A[2, 1]
            + A[0, 1] * A[1, 0] * A[2, 2]
        )
        return soma_positiva - soma_negativa

    # Ordem 4 ou maior: Laplace pela primeira linha.
    resultado = 0.0
    for j in range(n):
        submatriz = np.delete(np.delete(A, 0, axis=0), j, axis=1)
        cofator = (-1) ** j * determinante(submatriz)
        resultado += A[0, j] * cofator

    return resultado


## 3. Comparação com o NumPy

A função auxiliar abaixo apenas organiza a apresentação dos testes: mostra a matriz, os dois determinantes, o erro absoluto e a comparação com tolerância. `np.linalg.det` aparece aqui como referência, separada da função implementada.

Em ponto flutuante, o resultado de uma operação pode ter pequenos resíduos de arredondamento. Por isso usamos `np.isclose`, em vez de exigir igualdade exata com `==`. Com as tolerâncias escolhidas, aceitamos:

$$|d_{\mathrm{próprio}}-d_{\mathrm{NumPy}}|\leq 10^{-9}+10^{-9}|d_{\mathrm{NumPy}}|.$$

As tolerâncias `rtol=1e-9` e `atol=1e-9` são adequadas aos valores pequenos usados neste experimento. Não são uma garantia universal para toda matriz ou escala de valores. O `assert` interrompe a execução se um teste falhar.

**Base:** Brownlee, seção 11.5, pp. 84–85 (pp. 100–101 do PDF), que apresenta o cálculo de determinante com NumPy e um exemplo de resíduo numérico para uma matriz singular.


In [3]:
def comparar_determinantes(A):
    resultado = determinante(A)
    referencia = np.linalg.det(A)
    confere = np.isclose(resultado, referencia, rtol=1e-9, atol=1e-9)

    print("Matriz:")
    print(A)
    print("Determinante implementado:", resultado)
    print("Determinante NumPy:", referencia)
    print(f"Diferença absoluta em relação ao NumPy: {abs(resultado - referencia):.3e}")
    print("Concordam dentro da tolerância:", confere)
    assert confere, "O resultado difere da referência."


## 4. Teste com matriz de ordem 2

Usamos a matriz da **Week 5, p. 19**. Pela diferença entre os produtos das diagonais:

$$\det(A_2)=5\cdot(-3)-2\cdot4=-15-8=-23.$$

Esse teste executa o ramo `n == 2` e verifica também um determinante negativo.


In [4]:
A2 = np.array([[5, 2],
               [4, -3]])

comparar_determinantes(A2)
assert np.isclose(determinante(A2), -23, rtol=1e-9, atol=1e-9)


Matriz:
[[ 5  2]
 [ 4 -3]]
Determinante implementado: -23.0
Determinante NumPy: -23.0
Diferença absoluta em relação ao NumPy: 0.000e+00
Concordam dentro da tolerância: True


## 5. Teste com matriz de ordem 3

A matriz aparece na **Week 5, pp. 20–21**. Pela Regra de Sarrus:

$$S_+=4\cdot4\cdot0+2\cdot3\cdot1+(-1)\cdot5\cdot3=0+6-15=-9,$$
$$S_-=(-1)\cdot4\cdot1+4\cdot3\cdot3+2\cdot5\cdot0=-4+36+0=32,$$
$$\det(A_3)=S_+-S_-=-9-32=-41.$$

As expressões `soma_positiva` e `soma_negativa` agrupam os produtos conforme seus sinais na fórmula; os próprios valores dos produtos podem ser negativos. A Regra de Sarrus é usada somente na ordem 3.


In [5]:
A3 = np.array([[4, 2, -1],
               [5, 4, 3],
               [1, 3, 0]])

comparar_determinantes(A3)
assert np.isclose(determinante(A3), -41, rtol=1e-9, atol=1e-9)


Matriz:
[[ 4  2 -1]
 [ 5  4  3]
 [ 1  3  0]]
Determinante implementado: -41.0
Determinante NumPy: -41.00000000000001
Diferença absoluta em relação ao NumPy: 7.105e-15
Concordam dentro da tolerância: True


## 6. Teste com matriz de ordem 4

A matriz é a da **Week 5, pp. 23 e 26**, cujo determinante é **34**. A função aplica Laplace pela primeira linha, com sinais $+, -, +, -$. Cada submatriz resultante tem ordem 3 e é resolvida por Sarrus.

A expansão pode ser feita por qualquer linha ou coluna. Fixar a primeira linha simplifica o código. Uma linha com mais zeros poderia reduzir cálculos se os termos nulos fossem ignorados; aqui mantemos a expansão direta para facilitar o estudo.


In [6]:
A4 = np.array([[3, 1, 0, 1],
               [0, -1, 3, 4],
               [1, 1, 0, 2],
               [0, 1, 1, -1]])

comparar_determinantes(A4)
assert np.isclose(determinante(A4), 34, rtol=1e-9, atol=1e-9)


Matriz:
[[ 3  1  0  1]
 [ 0 -1  3  4]
 [ 1  1  0  2]
 [ 0  1  1 -1]]
Determinante implementado: 34.0
Determinante NumPy: 34.00000000000001
Diferença absoluta em relação ao NumPy: 7.105e-15
Concordam dentro da tolerância: True


## 7. Visualização dos passos de Laplace

Repetimos a expansão de $A_4$ mostrando cada submatriz, seu determinante, o cofator e a contribuição para a soma. Esta célula torna visível o mesmo processo usado dentro da função.

`axis=0` remove uma linha e `axis=1` remove uma coluna. O `j + 1` da impressão é apenas a numeração matemática das colunas; o cálculo continua usando índices iniciados em zero.


In [7]:
soma = 0.0

for j in range(A4.shape[0]):
    submatriz = np.delete(np.delete(A4, 0, axis=0), j, axis=1)
    det_menor = determinante(submatriz)
    cofator = (-1) ** j * det_menor
    termo = A4[0, j] * cofator
    soma += termo

    print(f"Coluna {j + 1}: elemento = {A4[0, j]}")
    print("Submatriz:")
    print(submatriz)
    print(f"Determinante da submatriz = {det_menor}")
    print(f"Cofator = {cofator}; contribuição = {termo}\n")

print("Soma das contribuições:", soma)
assert np.isclose(soma, 34, rtol=1e-9, atol=1e-9)


Coluna 1: elemento = 3
Submatriz:
[[-1  3  4]
 [ 1  0  2]
 [ 1  1 -1]]
Determinante da submatriz = 15.0
Cofator = 15.0; contribuição = 45.0

Coluna 2: elemento = 1
Submatriz:
[[ 0  3  4]
 [ 1  0  2]
 [ 0  1 -1]]
Determinante da submatriz = 7.0
Cofator = -7.0; contribuição = -7.0

Coluna 3: elemento = 0
Submatriz:
[[ 0 -1  4]
 [ 1  1  2]
 [ 0  1 -1]]
Determinante da submatriz = 3.0
Cofator = 3.0; contribuição = 0.0

Coluna 4: elemento = 1
Submatriz:
[[ 0 -1  3]
 [ 1  1  0]
 [ 0  1  1]]
Determinante da submatriz = 4.0
Cofator = -4.0; contribuição = -4.0

Soma das contribuições: 34.0


## 8. Teste com matriz de ordem 5

Usamos a matriz da **Week 5, p. 28**. A aula apresenta o exercício; neste notebook, o resultado calculado é **−138**. Laplace é aplicado primeiro à matriz 5 × 5 e depois às submatrizes 4 × 4. Os determinantes das submatrizes 3 × 3 são calculados por Sarrus.

Pela primeira linha, os determinantes das cinco submatrizes são $-304$, $-280$, $98$, $196$ e $-16$. Assim:

$$\det(A_5)=4(-304)-7(-280)+3(98)-6(196)+0(-16)=-138.$$

Este teste verifica a recursão em mais de um nível e compara o resultado com o NumPy.


In [8]:
A5 = np.array([[4, 7, 3, 6, 0],
               [1, 3, 4, 4, 9],
               [0, 0, 2, 1, 0],
               [1, 4, 2, 5, 2],
               [4, 3, 4, 0, 1]])

comparar_determinantes(A5)
assert np.isclose(determinante(A5), -138, rtol=1e-9, atol=1e-9)


Matriz:
[[4 7 3 6 0]
 [1 3 4 4 9]
 [0 0 2 1 0]
 [1 4 2 5 2]
 [4 3 4 0 1]]
Determinante implementado: -138.0
Determinante NumPy: -138.00000000000017
Diferença absoluta em relação ao NumPy: 1.705e-13
Concordam dentro da tolerância: True


## 9. Testes complementares

Além dos quatro testes pedidos, conferimos propriedades que ajudam a detectar erros de sinal e de recursão:

- Ordem 1: o determinante é o único elemento.
- Identidade: o determinante é 1.
- Matriz triangular: o determinante é o produto da diagonal principal.
- Duas linhas iguais: o determinante é zero.
- Troca de duas linhas: o determinante muda de sinal.

Os valores esperados são conhecidos por essas propriedades, oferecendo uma conferência adicional à comparação com o NumPy. São exemplos de teste, não uma prova de correção para todas as entradas.

**Base:** Robbiano, seção 4.6, pp. 95–96 do livro (pp. 114–115 do PDF); os cálculos também podem ser conferidos diretamente com as fórmulas usadas aqui.


In [9]:
casos = [
    ("Ordem 1", np.array([[7]]), 7),
    ("Identidade 5 x 5", np.eye(5), 1),
    ("Triangular 4 x 4", np.array([[2, 1, 3, 4],
                                  [0, -3, 2, 1],
                                  [0, 0, 4, 2],
                                  [0, 0, 0, 5]]), -120),
    ("Duas linhas iguais", np.array([[1, 2, 3],
                                    [1, 2, 3],
                                    [4, 5, 6]]), 0),
    ("Troca de linhas de A4", A4[[1, 0, 2, 3], :], -34),
]

for nome, A, esperado in casos:
    resultado = determinante(A)
    referencia = np.linalg.det(A)
    confere = (
        np.isclose(resultado, esperado, rtol=1e-9, atol=1e-9)
        and np.isclose(resultado, referencia, rtol=1e-9, atol=1e-9)
    )
    print(f"{nome}: resultado = {resultado}; esperado = {esperado}; confere = {confere}")
    assert confere, f"Falha no caso: {nome}"


Ordem 1: resultado = 7.0; esperado = 7; confere = True
Identidade 5 x 5: resultado = 1.0; esperado = 1; confere = True
Triangular 4 x 4: resultado = -120.0; esperado = -120; confere = True
Duas linhas iguais: resultado = 0.0; esperado = 0; confere = True
Troca de linhas de A4: resultado = -34.0; esperado = -34; confere = True


## 10. Verificação de entradas inválidas

Um vetor, uma matriz retangular e uma matriz de ordem zero não pertencem ao domínio adotado neste experimento. A função deve rejeitar essas entradas com `ValueError`, em vez de calcular um valor sem significado para o exercício. O bloco `try` / `except` permite mostrar cada mensagem sem interromper os demais testes.

A convenção matemática para o determinante da matriz vazia fica fora do escopo; aqui aceitamos somente ordens positivas. Também não tratamos números complexos, valores infinitos ou `NaN`.


In [10]:
entradas_invalidas = [
    ("Vetor", np.array([1, 2, 3])),
    ("Matriz retangular", np.array([[1, 2, 3], [4, 5, 6]])),
    ("Matriz vazia", np.empty((0, 0))),
]

for nome, entrada in entradas_invalidas:
    try:
        determinante(entrada)
    except ValueError as erro:
        print(f"{nome}: rejeição esperada ({erro})")
    else:
        raise AssertionError(f"A entrada deveria ter sido rejeitada: {nome}")


Vetor: rejeição esperada (A matriz deve ser quadrada.)
Matriz retangular: rejeição esperada (A matriz deve ser quadrada.)
Matriz vazia: rejeição esperada (A matriz não pode ser vazia.)


## 11. Conclusão e referências

Nos testes principais, os determinantes obtidos foram **−23**, **−41**, **34** e **−138**, respectivamente para as ordens 2, 3, 4 e 5. As comparações com `np.linalg.det` ficaram dentro das tolerâncias adotadas. Os testes complementares e de entradas inválidas também passaram na execução salva.

A implementação segue os três métodos exigidos. A recursão de Laplace reduz a ordem em cada chamada e chega ao caso direto de Sarrus. Embora a função aceite qualquer ordem positiva em sua definição, a expansão direta tem crescimento fatorial de trabalho e torna-se cara para matrizes grandes. O uso de `float` também limita a precisão dos resultados. O objetivo aqui é estudar os métodos, com exemplos pequenos.

#### Referências:

1. **Material da disciplina, Week 5**, pp. 17–29 do PDF: definição de determinante, produto das diagonais, Sarrus, Laplace e matrizes dos quatro testes. A matriz 5 × 5 está na p. 28; o valor −138 foi calculado neste experimento.
2. **ROBBIANO, Lorenzo. Álgebra Linear para Todos**, seção 3.7, pp. 73–75 do livro (pp. 92–94 do PDF): determinantes, submatrizes e expansão por cofatores; seção 4.6, pp. 95–96 (pp. 114–115 do PDF): propriedades e custo do cálculo.
3. **BROWNLEE, Jason. Basics of Linear Algebra for Machine Learning**, seção 11.5, pp. 84–85 do livro (pp. 100–101 do PDF): determinante com NumPy e arredondamento numérico.
